In [ ]:
!pip install rdflib pydotplus

In [ ]:
import base64
import json
import os
from pathlib import Path
import io

import requests
from PIL import Image
import rdflib
import pydotplus
from rdflib.tools.rdf2dot import rdf2dot

# ---DEPENDENCY---
# pip install requests pillow rdflib pydotplus

# 1. CONFIGURATION
def load_api_key(name, fallback_names=()):
    import os
    import sys
    from pathlib import Path

    candidates = (name, *fallback_names)
    repo_root = next(
        (
            candidate
            for candidate in (Path.cwd(), *Path.cwd().parents)
            if (candidate / "common" / "__init__.py").exists()
        ),
        None,
    )

    get_api_key = None
    if repo_root is not None:
        if str(repo_root) not in sys.path:
            sys.path.insert(0, str(repo_root))
        try:
            from common import get_api_key as shared_get_api_key
        except ImportError:
            pass
        else:
            get_api_key = shared_get_api_key

    if get_api_key is not None:
        for candidate in candidates:
            try:
                return get_api_key(candidate)
            except KeyError:
                pass

    for candidate in candidates:
        value = os.getenv(candidate, "").strip()
        if value:
            return value

    joined = ", ".join(candidates)
    raise RuntimeError(
        f"Missing API key. Set one of [{joined}] in the environment"
        " or the top-level config file."
    )

GEMINI_API_KEY = load_api_key("GEMINI_API_KEY")

MODEL = "gemini-2.5-flash"
ENDPOINT = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent?key={GEMINI_API_KEY}"


# 2. HELPER FUNCTIONS
def encode_image_to_base64(image_path: str) -> str:
    
    try:
        with Image.open(image_path) as img:
            img = img.convert("RGB")
            buffer = Path(image_path).read_bytes()
            return base64.b64encode(buffer).decode("utf-8")
    except FileNotFoundError:
        print(f"Error: Image file not found at {image_path}")
        return None
    except Exception as e:
        print(f"An error occurred while processing the image: {e}")
        return None

def build_request(image_b64: str, prompt: str) -> dict:
    
    # JSON payload for the Gemini API multimodal request.
    
    return {
        "contents": [
            {
                "role": "user",
                "parts": [
                    {"inlinedata": {"mimeType": "image/jpeg", "data": image_b64}},
                    {"text": prompt},
                ],
            }
        ],
        "generationConfig": {
            "temperature": 0.1,
            "topP": 0.9,
            "maxOutputTokens": 8192,
        },
    }

def call_gemini(payload: dict) -> dict:
    headers = {"Content-Type": "application/json"}
    try:
        response = requests.post(ENDPOINT, headers=headers, data=json.dumps(payload))
        response.raise_for_status()
        return response.json()
    except requests.exceptions.HTTPError as e:
        print(f"An HTTP error occurred: {e}")
        print(f"Response body: {e.response.text}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

def save_graph_visualization(rdf_data: str, output_filename: str = "knowledge_graph.png"):
    """
    Parses RDF data, generates a visual graph, and saves it as a PNG file.
    """
    # Create an rdflib Graph 
    graph = rdflib.Graph()

    try:
        # Parse the Turtle RDF 
        graph.parse(data=rdf_data, format="turtle")
        print(f"Successfully parsed {len(graph)} triples.")

        # Create an in-memory stream to hold the DOT graph data
        stream = io.StringIO()

        # Convert the rdflib graph to DOT
        rdf2dot(graph, stream, opts={})

        # Use pydotplus to parse the DOT data and create the image
        pydot_graph = pydotplus.graph_from_dot_data(stream.getvalue())
        pydot_graph.write_png(output_filename)

        print(f"--- Knowledge Graph visualization saved to {output_filename} ---")

    except Exception as e:
        print(f"An error occurred during graph parsing or visualization: {e}")


# 3. MAIN EXECUTION
if __name__ == "__main__":
    # input image
    img_path = "/content/table_2.1.jpg"
    output_rdf_filename = "output.ttl"
    output_image_filename = "knowledge_graph.png"

    prompt_text = (
        "You are an expert knowledge engineer. Your task is to extract data from the image "
        "and convert it into a formal RDF knowledge graph using Turtle (.ttl) syntax. "
        "Strictly adhere to the following schema and best practices:\n\n"
        "## RDF Schema & Vocabulary:\n"
        "1. Use the provided prefixes: `soil`, `skos`, `rdf`, `rdfs`, `dcterms`.\n"
        "2. Formally define `soil:SocietalNeed` and `soil:SoilFunction` as `rdfs:Class`.\n"
        "3. Formally define `soil:hasSoilFunction` as an `rdf:Property`.\n"
        "4. Model each specific item (e.g., 'Biomass', 'Water') as an instance of its class.\n"
        "5. For each instance, provide a `skos:prefLabel` with its name.\n"
        "6. Link Societal Need instances to Soil Function instances using `soil:hasSoilFunction`.\n\n"
        "## Output Template:\n"
        "Return only the Turtle code. Do not include any other explanations.\n\n"
        "```turtle\n"
        "@prefix soil: [http://example.org/soil-health/ontology/](http://example.org/soil-health/ontology/) .\n"
        "@prefix skos: [http://www.w3.org/2004/02/skos/core#](http://www.w3.org/2004/02/skos/core#) .\n"
        "@prefix rdf: [http://www.w3.org/1999/02/22-rdf-syntax-ns#](http://www.w3.org/1999/02/22-rdf-syntax-ns#) .\n"
        "@prefix rdfs: [http://www.w3.org/2000/01/rdf-schema#](http://www.w3.org/2000/01/rdf-schema#) .\n"
        "@prefix dcterms: [http://purl.org/dc/terms/](http://purl.org/dc/terms/) .\n\n"
        "# ... Ontology and data ...\n"
        "```"
    )

    print(f"Processing image: {img_path}")
    image_b64 = encode_image_to_base64(img_path)

    if image_b64:
        request_body = build_request(image_b64, prompt_text)
        print("Sending request to Gemini API...")
        result = call_gemini(request_body)

        if result and "candidates" in result:
            # Extract the raw text from the response
            extracted_rdf = result["candidates"][0]["content"]["parts"][0]["text"]

            # Clean the response to ensure 
            if extracted_rdf.strip().startswith("```turtle"):
                extracted_rdf = extracted_rdf.strip()[len("```turtle"):-len("```")].strip()
            elif extracted_rdf.strip().startswith("```"):
                extracted_rdf = extracted_rdf.strip()[len("```"):-len("```")].strip()

            print("\n--- Gemini RDF Output ---")
            print(extracted_rdf)

            # Save the RDF output
            with open(output_rdf_filename, "w", encoding="utf-8") as f:
                f.write(extracted_rdf)
            print(f"\n--- Successfully saved RDF to {output_rdf_filename} ---")

            # save graph as an image
            print("\n--- Starting knowledge graph output ---")
            save_graph_visualization(extracted_rdf, output_image_filename)

        else:
            print("Failed to get a valid response from the Gemini API.")

